Assignemtn 4 - LSTM (test)

In [1]:
from torch import nn, optim
from torch.nn import functional as F

import os
import torch
import spacy

torch.manual_seed(1)

In [2]:
# stepwise
lstm = nn.LSTM(3, 3)
inputs = [torch.randn(1, 3) for _ in range(5)]

# initialize the hidden state
hidden = (torch.randn(1, 1, 3), torch.randn(1, 1, 3))

for i in inputs:
    out, hidden = lstm(i.view(1, 1, -1), hidden)

out, hidden

(tensor([[[-0.3600,  0.0893,  0.0215]]], grad_fn=<MkldnnRnnLayerBackward0>),
 (tensor([[[-0.3600,  0.0893,  0.0215]]], grad_fn=<StackBackward0>),
  tensor([[[-1.1298,  0.4467,  0.0254]]], grad_fn=<StackBackward0>)))

In [3]:
# Sequence at once, whole sequence
lstm = nn.LSTM(3, 3)
inputs = [torch.randn(1, 3) for _ in range(5)]

inputs = torch.cat(inputs).view(len(inputs), 1, -1)
hidden = (torch.randn(1, 1, 3), torch.randn(1, 1, 3)) # initialize the hidden state

out, hidden = lstm(inputs, hidden)

print(out)
print(hidden)

tensor([[[-0.2696,  0.2599, -0.0758]],

        [[-0.4923,  0.1408, -0.0738]],

        [[-0.4523,  0.1241, -0.1461]],

        [[-0.3057,  0.1198, -0.0571]],

        [[-0.1077,  0.0289, -0.0487]]], grad_fn=<MkldnnRnnLayerBackward0>)
(tensor([[[-0.1077,  0.0289, -0.0487]]], grad_fn=<StackBackward0>), tensor([[[-0.1439,  0.1426, -0.2563]]], grad_fn=<StackBackward0>))


In [4]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, num_labels):
        super(LSTMClassifier, self).__init__()
        self.hidden_dim = hidden_dim

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim.
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        # The linear layer that maps from hidden state space to tag space
        self.classify = nn.Linear(hidden_dim, num_labels)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.classify(lstm_out.view(len(sentence), -1))
        scores = F.log_softmax(tag_space, dim=1)
        return scores

In [5]:
datadir = r'./data/tweeteval/datasets/emotion'

In [6]:
def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)


training_data = [
    # Tags are: DET - determiner; NN - noun; V - verb
    # For example, the word "The" is a determiner
    ("The dog ate the apple".split(), ["DET", "NN", "V", "DET", "NN"]),
    ("Everybody read that book".split(), ["NN", "V", "DET", "NN"])
]
word_to_ix = {}
# For each words-list (sentence) and tags-list in each tuple of training_data
for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix:  # word has not been assigned an index yet
            word_to_ix[word] = len(word_to_ix)  # Assign each word with a unique index
print(word_to_ix)
tag_to_ix = {"DET": 0, "NN": 1, "V": 2}  # Assign each tag with a unique index

# These will usually be more like 32 or 64 dimensional.
# We will keep them small, so we can see how the weights change as we train.


{'The': 0, 'dog': 1, 'ate': 2, 'the': 3, 'apple': 4, 'Everybody': 5, 'read': 6, 'that': 7, 'book': 8}


In [30]:
nlp = spacy.load("en_core_web_sm")

def tokenize_no_stopwords(text):
    doc = nlp(text)
    pruned = ' '.join(t.text for t in doc if not t.is_stop)
    # return list(t.text for t in doc if not t.is_stop)
    return pruned

def get_training_data():
    fpath_x = os.path.join(datadir, 'train_text.txt')
    fpath_y = os.path.join(datadir, 'train_labels.txt')
    with open(fpath_x, 'r', encoding='utf-8') as f:
        rows = [l.strip() for l in f.readlines()]
    with open(fpath_y, 'r', encoding='utf-8') as f:
        labels = [l.strip() for l in f.readlines()]
    return list(zip(rows, labels))

for x, y in get_training_data()[:5]:
    print(f'Original:{x}\nPruned:{tokenize_no_stopwords(x)}\n{y=}\n')


Original:“Worry is a down payment on a problem you may never have'.  Joyce Meyer.  #motivation #leadership #worry
Pruned:“ Worry payment problem ' .   Joyce Meyer .   # motivation # leadership # worry
y='2'

Original:My roommate: it's okay that we can't spell because we have autocorrect. #terrible #firstworldprobs
Pruned:roommate : okay spell autocorrect . # terrible # firstworldprobs
y='0'

Original:No but that's so cute. Atsu was probably shy about photos before but cherry helped her out uwu
Pruned:cute . Atsu probably shy photos cherry helped uwu
y='1'

Original:Rooneys fucking untouchable isn't he? Been fucking dreadful again, depay has looked decent(ish)tonight
Pruned:Rooneys fucking untouchable ? fucking dreadful , depay looked decent(ish)tonight
y='0'

Original:it's pretty depressing when u hit pan on ur favourite highlighter
Pruned:pretty depressing u hit pan ur favourite highlighter
y='3'



In [10]:
training_data = get_training_data()
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
VOCAB_SIZE = 3000

model = LSTMClassifier(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    vocab_size=VOCAB_SIZE,
    num_labels=4
)

loss_function = nn.NLLLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# See what the scores are before training
# Note that element i,j of the output is the score for tag j for word i.
# Here we don't need to train, so the code is wrapped in torch.no_grad()
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)
    print(tag_scores)

for epoch in range(300):  # again, normally you would NOT do 300 epochs, it is toy data
    for sentence, tags in training_data:
        # Step 1. Remember that Pytorch accumulates gradients.
        # We need to clear them out before each instance
        model.zero_grad()

        # Step 2. Get our inputs ready for the network, that is, turn them into
        # Tensors of word indices.
        sentence_in = prepare_sequence(sentence, word_to_ix)
        targets = prepare_sequence(tags, tag_to_ix)

        # Step 3. Run our forward pass.
        tag_scores = model(sentence_in)

        # Step 4. Compute the loss, gradients, and update the parameters by
        #  calling optimizer.step()
        loss = loss_function(tag_scores, targets)
        loss.backward()
        optimizer.step()

# See what the scores are after training
with torch.no_grad():
    inputs = prepare_sequence(training_data[0][0], word_to_ix)
    tag_scores = model(inputs)

    # The sentence is "the dog ate the apple".  i,j corresponds to score for tag j
    # for word i. The predicted tag is the maximum scoring tag.
    # Here, we can see the predicted sequence below is 0 1 2 0 1
    # since 0 is index of the maximum value of row 1,
    # 1 is the index of maximum value of row 2, etc.
    # Which is DET NOUN VERB DET NOUN, the correct sequence!
    print(tag_scores)

FileNotFoundError: [Errno 2] No such file or directory: './data/tweeteval/datasets/emotion\\train_text.txt'

In [13]:
nlp = spacy.load("en_core_web_sm")
doc = nlp("An Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print(token.text, f'{token.is_stop}')

' '.join(t.text for t in doc if not t.is_stop)


An True
Apple False
is True
looking False
at True
buying False
U.K. False
startup False
for True
$ False
1 False
billion False


'Apple looking buying U.K. startup $ 1 billion'